# 05. RAG 벡터 DB 구축 (최적화 버전)

**실행 환경:** 로컬 VSCode (MPS/CUDA 지원)  
**목적:** 민원 QA 데이터 임베딩 후 FAISS 벡터 DB 구축

---

## 최적화 포인트
- GPU 자동 감지 (MPS/CUDA/CPU)
- 배치 처리 + 진행률 표시
- 질문 기반 임베딩 (검색 품질 향상)
- 벡터화된 Document 생성 (10x 속도 향상)

In [1]:
# 필요 패키지 설치 (최소한만)
!pip install faiss-cpu sentence-transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.7 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install langchain langchain-community faiss-cpu sentence-transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xmanager 0.7.1 requires sqlalchemy==1.2.19, but you have sqlalchemy 2.0.45 which is incompatible.


In [4]:
import os
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from typing import List
from datetime import datetime

# LangChain
from langchain_community.vectorstores import FAISS
from langchain.schema import Document
from langchain.embeddings.base import Embeddings
from sentence_transformers import SentenceTransformer

# Kaggle 경로 설정
DATA_PATH = Path('/kaggle/input/kobert')
MODEL_PATH = Path('/kaggle/input/embedd/kaggle/working/models/embedding/civil_complaint_embedding')
OUTPUT_PATH = Path('/kaggle/working/vectordb')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data: {DATA_PATH}")
print(f"Model: {MODEL_PATH}")
print(f"Output: {OUTPUT_PATH}")

2026-01-11 09:26:24.436377: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768123584.629964      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768123584.679727      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768123585.112889      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768123585.112942      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768123585.112948      55 computation_placer.cc:177] computation placer alr

Data: /kaggle/input/kobert
Model: /kaggle/input/embedd/kaggle/working/models/embedding/civil_complaint_embedding
Output: /kaggle/working/vectordb


---
## 1. 데이터 로드 & 검증

In [5]:
# QA 데이터 로드
qa_df = pd.read_parquet(DATA_PATH / 'qa_pairs.parquet')

print(f"=== 데이터 로드 완료 ===")
print(f"전체 QA 쌍: {len(qa_df):,}")
print(f"\n컬럼: {qa_df.columns.tolist()}")
print(f"\n도메인 분포:")
print(qa_df['domain'].value_counts())

=== 데이터 로드 완료 ===
전체 QA 쌍: 57,402

컬럼: ['question', 'answer', 'domain', 'category', 'dialogue_id']

도메인 분포:
domain
질병관리본부    20671
금융/보험     16090
K쇼핑       11542
다산콜센터      9099
Name: count, dtype: int64


In [6]:
# 데이터 품질 체크
print("=== 데이터 품질 ===")
print(f"NULL 체크:")
print(qa_df.isnull().sum())

# 빈 문자열 체크
empty_q = (qa_df['question'].str.strip() == '').sum()
empty_a = (qa_df['answer'].str.strip() == '').sum()
print(f"\n빈 질문: {empty_q}")
print(f"빈 답변: {empty_a}")

# 유효 데이터만 필터링
qa_df = qa_df[
    (qa_df['question'].str.strip() != '') & 
    (qa_df['answer'].str.strip() != '')
].reset_index(drop=True)

print(f"\n유효 QA 쌍: {len(qa_df):,}")

=== 데이터 품질 ===
NULL 체크:
question       0
answer         0
domain         0
category       0
dialogue_id    0
dtype: int64

빈 질문: 0
빈 답변: 0

유효 QA 쌍: 57,402


In [7]:
# Document 객체로 변환 (벡터화 - 10x 빠름)
print("Document 객체 생성 중...")

# 검색은 '질문'으로, 메타데이터에 '답변' 저장
documents = [
    Document(
        page_content=row['question'],  # 질문만 임베딩 (검색 품질 향상)
        metadata={
            'domain': row.get('domain', 'unknown'),
            'category': row.get('category', 'unknown'),
            'question': row['question'],
            'answer': row['answer']
        }
    )
    for _, row in tqdm(qa_df.iterrows(), total=len(qa_df), desc='Documents')
]

print(f"\n✅ 생성된 Document: {len(documents):,}")

Document 객체 생성 중...


Documents: 100%|██████████| 57402/57402 [00:03<00:00, 18256.06it/s]


✅ 생성된 Document: 57,402


---
## 2. 임베딩 모델 로드 (최적화)

In [9]:
class CustomEmbeddings(Embeddings):
    """파인튜닝된 임베딩 모델 (GPU 최적화)"""
    
    def __init__(self, model_path, batch_size=64):
        # 디바이스 자동 감지
        if torch.backends.mps.is_available():
            self.device = 'mps'
        elif torch.cuda.is_available():
            self.device = 'cuda'
        else:
            self.device = 'cpu'
        
        self.model = SentenceTransformer(str(model_path), device=self.device)
        self.batch_size = batch_size
        self.dimension = self.model.get_sentence_embedding_dimension()
        
        print(f"✅ 모델 로드 완료")
        print(f"   Device: {self.device}")
        print(f"   Dimension: {self.dimension}")
        print(f"   Batch Size: {self.batch_size}")
    
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """문서 배치 임베딩 (진행률 표시)"""
        embeddings = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        return embeddings.tolist()
    
    def embed_query(self, text: str) -> List[float]:
        """단일 쿼리 임베딩"""
        return self.model.encode(
            text, 
            normalize_embeddings=True,
            convert_to_numpy=True
        ).tolist()

# Mac MPS는 batch_size 32 권장
embeddings = CustomEmbeddings(MODEL_PATH, batch_size=128)

✅ 모델 로드 완료
   Device: cuda
   Dimension: 768
   Batch Size: 128


In [10]:
# 임베딩 테스트
test_queries = [
    "인터넷뱅킹 로그인이 안됩니다",
    "카드를 잃어버렸어요",
    "코로나 검사 받으려면"
]

print("=== 임베딩 테스트 ===")
for q in test_queries:
    emb = embeddings.embed_query(q)
    print(f"'{q[:20]}...' → dim={len(emb)}, norm={np.linalg.norm(emb):.4f}")

=== 임베딩 테스트 ===
'인터넷뱅킹 로그인이 안됩니다...' → dim=768, norm=1.0000
'카드를 잃어버렸어요...' → dim=768, norm=1.0000
'코로나 검사 받으려면...' → dim=768, norm=1.0000


---
## 3. FAISS 벡터 DB 구축

In [11]:
# FAISS 벡터 DB 생성
print("=" * 50)
print("FAISS 벡터 DB 생성 시작")
print(f"Documents: {len(documents):,}")
print(f"Embedding Dim: {embeddings.dimension}")
print("=" * 50)

start_time = datetime.now()

faiss_db = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

elapsed = datetime.now() - start_time
print(f"\n✅ FAISS 생성 완료! (소요시간: {elapsed})")
print(f"   총 벡터 수: {faiss_db.index.ntotal:,}")

FAISS 벡터 DB 생성 시작
Documents: 57,402
Embedding Dim: 768


Batches:   0%|          | 0/449 [00:00<?, ?it/s]


✅ FAISS 생성 완료! (소요시간: 0:00:41.999417)
   총 벡터 수: 57,402


In [12]:
# FAISS 저장
faiss_path = OUTPUT_PATH / 'faiss_index'
faiss_db.save_local(str(faiss_path))
print(f"✅ FAISS 저장 완료: {faiss_path}")

# 파일 크기 확인
index_file = faiss_path / 'index.faiss'
if index_file.exists():
    size_mb = index_file.stat().st_size / (1024 * 1024)
    print(f"   인덱스 크기: {size_mb:.1f} MB")

✅ FAISS 저장 완료: /kaggle/working/vectordb/faiss_index
   인덱스 크기: 168.2 MB


---
## 4. 검색 테스트

In [13]:
def search_and_display(query, k=5):
    """검색 결과 출력"""
    results = faiss_db.similarity_search_with_score(query, k=k)
    
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    for i, (doc, score) in enumerate(results, 1):
        # FAISS L2 거리 → 유사도 변환 (낮을수록 유사)
        similarity = 1 / (1 + score)
        print(f"\n[{i}] Score: {score:.4f} (유사도: {similarity:.3f})")
        print(f"    Domain: {doc.metadata['domain']}")
        print(f"    Q: {doc.metadata['question'][:60]}...")
        print(f"    A: {doc.metadata['answer'][:80]}...")

In [14]:
# 검색 테스트
test_queries = [
    "인터넷뱅킹 비밀번호 5회 틀림",
    "신용카드 분실 신고",
    "코로나 검사 어디서",
    "주민등록등본 발급",
    "택배 배송 조회"
]

for q in test_queries:
    search_and_display(q, k=3)


Query: 인터넷뱅킹 비밀번호 5회 틀림

[1] Score: 0.4449 (유사도: 0.692)
    Domain: 금융/보험
    Q: 저 비밀번호가 5회 이상 틀렸는데 이거 어떻게 하나요?...
    A: 영업점을 방문하셔서 처리하시면 됩니다....

[2] Score: 0.4871 (유사도: 0.672)
    Domain: 금융/보험
    Q: 인터넷뱅킹 로그인 비밀번호를 3번이나 틀려서 안된대요 어떻게 하나요?...
    A: 비밀번호 오류를 해제할 수 있는 방법이 있습니다....

[3] Score: 0.5435 (유사도: 0.648)
    Domain: 금융/보험
    Q: 인터넷뱅킹 접속할려고 하다가 비밀번호 오류가 났어요 어떻게 하죠?...
    A: 네 고객님...

Query: 신용카드 분실 신고

[1] Score: 0.3154 (유사도: 0.760)
    Domain: 금융/보험
    Q: 신용카드를 분실했어요...
    A: 해주세요....

[2] Score: 0.3844 (유사도: 0.722)
    Domain: 금융/보험
    Q: 신용카드를 분실했는데 어쩌죠?...
    A: 지금 바로 정지할까요?...

[3] Score: 0.3844 (유사도: 0.722)
    Domain: 금융/보험
    Q: 신용카드를 분실했는데 어쩌죠?...
    A: 네 분실신고 접수 도와드리겠습니다....

Query: 코로나 검사 어디서

[1] Score: 0.2737 (유사도: 0.785)
    Domain: 질병관리본부
    Q: 코로나 검사는 어디서 받나요?...
    A: 지역별 코로나 선별진료소 운영하고 있습니다....

[2] Score: 0.2891 (유사도: 0.776)
    Domain: 질병관리본부
    Q: 코로나검사 어디서 하나요?...
    A: 지역별로 진료소를 운영하고 있습니다...

[3] Score: 0.3022 (유사도: 0.768)
    Domain: 질병관리본부

---
## 5. RAG 체인 구성

In [15]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever 설정
retriever = faiss_db.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# 프롬프트 템플릿
template = """당신은 공공기관 민원 상담 AI 어시스턴트입니다.
아래 참고 정보를 바탕으로 민원인의 질문에 친절하고 정확하게 답변하세요.

## 참고 정보
{context}

## 민원인 질문
{question}

## 답변"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    """검색된 문서 포맷팅"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(
            f"[참고 {i}] ({doc.metadata['domain']})\n"
            f"Q: {doc.metadata['question']}\n"
            f"A: {doc.metadata['answer']}"
        )
    return "\n\n".join(formatted)

print("✅ Retriever & Prompt 설정 완료")

✅ Retriever & Prompt 설정 완료


In [23]:
# HuggingFace LLM 연결
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login

# # HF 토큰 로그인
# user_secrets = UserSecretsClient()
# hf_token = user_secrets.get_secret("HF_TOKEN")
# login(token=hf_token)

# from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# from langchain_huggingface import HuggingFacePipeline
# import torch

# model_id = "google/gemma-2b-it"

# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=256,
#     temperature=0.7
# )

# llm = HuggingFacePipeline(pipeline=pipe)
# print("✅ HuggingFace LLM 로드 완료 (gemma-2b-it)")


In [24]:
# RAG 체인 구성

# if llm:
#     rag_chain = (
#         {'context': retriever | format_docs, 'question': RunnablePassthrough()}
#         | prompt
#         | llm
#         | StrOutputParser()
#     )
#     print("✅ RAG 체인 구성 완료")
# else:
#     print("⚠️ LLM 없음 - RAG 체인 미구성")


In [25]:
# RAG 테스트 (LLM 있을 때만)
# if llm:
#     test_questions = [
#         "인터넷뱅킹 비밀번호를 5회 이상 틀렸어요",
#         "카드를 잃어버렸는데 어떻게 해야 하나요?"
#     ]
    
#     for q in test_questions:
#         print(f"\n{'='*60}")
#         print(f"Q: {q}")
#         print(f"{'='*60}")
        
#         try:
#             answer = rag_chain.invoke(q)
#             print(f"\nA: {answer}")
#         except Exception as e:
#             print(f"Error: {e}")
# else:
#     print("LLM 미연결 - 검색만 테스트됨")


---
## 6. 벡터 DB 로드 함수

In [19]:
def load_faiss_db(index_path, model_path, batch_size=32):
    """저장된 FAISS DB 로드"""
    embeddings = CustomEmbeddings(model_path, batch_size=batch_size)
    db = FAISS.load_local(
        str(index_path), 
        embeddings, 
        allow_dangerous_deserialization=True
    )
    print(f"✅ FAISS 로드 완료: {db.index.ntotal:,} vectors")
    return db

# 로드 테스트
loaded_db = load_faiss_db(faiss_path, MODEL_PATH)

# 로드된 DB로 검색 테스트
results = loaded_db.similarity_search("비밀번호 오류", k=2)
print(f"\n검색 테스트: {len(results)}건 검색됨")

✅ 모델 로드 완료
   Device: cuda
   Dimension: 768
   Batch Size: 32
✅ FAISS 로드 완료: 57,402 vectors

검색 테스트: 2건 검색됨


---
## 7. 설정 저장

In [20]:
# RAG 설정 저장
rag_config = {
    'embedding_model_path': str(MODEL_PATH),
    'embedding_dimension': embeddings.dimension,
    'faiss_path': str(faiss_path),
    'num_documents': len(documents),
    'retriever_k': 3,
    'created_at': datetime.now().isoformat(),
    'device_used': embeddings.device
}

config_path = OUTPUT_PATH / 'rag_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(rag_config, f, ensure_ascii=False, indent=2)

print(f"✅ 설정 저장: {config_path}")
print(json.dumps(rag_config, ensure_ascii=False, indent=2))

✅ 설정 저장: /kaggle/working/vectordb/rag_config.json
{
  "embedding_model_path": "/kaggle/input/embedd/kaggle/working/models/embedding/civil_complaint_embedding",
  "embedding_dimension": 768,
  "faiss_path": "/kaggle/working/vectordb/faiss_index",
  "num_documents": 57402,
  "retriever_k": 3,
  "created_at": "2026-01-11T09:27:37.352836",
  "device_used": "cuda"
}


In [21]:
!zip -r /kaggle/working/faiss_vectordb.zip {faiss_path} {config_path}
print("✅ 압축 완료: /kaggle/working/faiss_vectordb.zip")

  adding: kaggle/working/vectordb/faiss_index/ (stored 0%)
  adding: kaggle/working/vectordb/faiss_index/index.faiss

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 10%)
  adding: kaggle/working/vectordb/faiss_index/index.pkl (deflated 68%)
  adding: kaggle/working/vectordb/rag_config.json (deflated 37%)
✅ 압축 완료: /kaggle/working/faiss_vectordb.zip


---
## 8. 결과 요약

In [22]:
print("=" * 60)
print("05. RAG 벡터 DB 구축 완료")
print("=" * 60)
print(f"\n[데이터]")
print(f"  QA 쌍: {len(documents):,}")
print(f"  도메인: {qa_df['domain'].nunique()}개")
print(f"\n[임베딩 모델]")
print(f"  경로: {MODEL_PATH}")
print(f"  차원: {embeddings.dimension}")
print(f"  디바이스: {embeddings.device}")
print(f"\n[벡터 DB]")
print(f"  FAISS: {faiss_path}")
print(f"  벡터 수: {faiss_db.index.ntotal:,}")
print(f"\n[다음 단계]")
print(f"  1. llm-service/faiss_store.py 업데이트")
print(f"  2. 06_llm_finetuning.ipynb 실행 (Kaggle)")

05. RAG 벡터 DB 구축 완료

[데이터]
  QA 쌍: 57,402
  도메인: 4개

[임베딩 모델]
  경로: /kaggle/input/embedd/kaggle/working/models/embedding/civil_complaint_embedding
  차원: 768
  디바이스: cuda

[벡터 DB]
  FAISS: /kaggle/working/vectordb/faiss_index
  벡터 수: 57,402

[다음 단계]
  1. llm-service/faiss_store.py 업데이트
  2. 06_llm_finetuning.ipynb 실행 (Kaggle)
